# EM Algorithm: Inferring Hidden Coin Bias

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/ml-tutorials/blob/main/notebooks/bayesian/em_coin_toss.ipynb)

**Companion blog post:** [The EM Algorithm: An Intuitive Guide with the Coin Toss Example](https://sesen.ai/blog/em-algorithm-coin-toss-intuitive-guide)

---

In this notebook, you'll implement the Expectation-Maximisation (EM) algorithm from scratch to solve a classic problem: inferring the bias of two coins when you don't know which coin was used for each experiment.

By the end, you'll understand:
- How EM iteratively improves parameter estimates
- The intuition behind "soft assignments" (E-step)
- How weighted maximum likelihood works (M-step)
- Why EM always converges (though not necessarily to the global optimum)

## 1. The Problem

Imagine you have **two biased coins** (A and B) with unknown probabilities of landing heads.

Someone conducts **5 experiments**, each consisting of 10 coin tosses. For each experiment, they:
1. Secretly pick either Coin A or Coin B
2. Toss that coin 10 times
3. Record only the number of heads (not which coin they used!)

You observe:
- Experiment 1: 5 heads, 5 tails
- Experiment 2: 9 heads, 1 tail
- Experiment 3: 8 heads, 2 tails
- Experiment 4: 4 heads, 6 tails
- Experiment 5: 7 heads, 3 tails

**Your task:** Estimate the bias (probability of heads) for each coin.

If you knew which coin was used, this would be trivial (just count heads/total). But the coin identity is **hidden** - this is what makes it an EM problem.

## 2. Quick Win: Run the Complete Algorithm

Let's run the complete EM algorithm first. You'll see the coin biases converge, and then we'll understand how it works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log, factorial, exp

def get_log_likelihood(obs, probs):
    """
    Compute log-likelihood of observations given coin probabilities.
    
    Uses the multinomial distribution:
    ln[P(obs|probs)] = ln(n!) - sum(ln(x_i!)) + sum(x_i * ln(p_i))
    
    Args:
        obs: tuple of (heads, tails) counts
        probs: tuple of (p_heads, p_tails) probabilities
    
    Returns:
        Log-likelihood value
    """
    n = sum(obs)
    log_multinomial_coeff = log(factorial(n)) - sum(log(factorial(x)) for x in obs)
    log_prob = sum(x * log(p) for x, p in zip(obs, probs))
    return log_multinomial_coeff + log_prob


def em_coin_toss(head_counts, n_tosses=10, theta_A_init=0.6, theta_B_init=0.5, 
                 delta=0.001, max_iter=100):
    """
    EM algorithm for inferring two coin biases from unlabelled experiments.
    
    Args:
        head_counts: array of head counts for each experiment
        n_tosses: number of tosses per experiment
        theta_A_init: initial guess for P(heads | coin A)
        theta_B_init: initial guess for P(heads | coin B)
        delta: convergence threshold
        max_iter: maximum iterations
    
    Returns:
        theta_A_history, theta_B_history: parameter values at each iteration
    """
    # Prepare experiments as (heads, tails) tuples
    tail_counts = n_tosses - head_counts
    experiments = list(zip(head_counts, tail_counts))
    
    # Initialise parameter histories
    theta_A = [theta_A_init]
    theta_B = [theta_B_init]
    
    iteration = 0
    improvement = float('inf')
    
    while improvement > delta and iteration < max_iter:
        current_theta_A = theta_A[-1]
        current_theta_B = theta_B[-1]
        
        # E-STEP: Compute "soft assignments" - probability each experiment came from each coin
        expected_A = np.zeros((len(experiments), 2))
        expected_B = np.zeros((len(experiments), 2))
        
        for i, exp in enumerate(experiments):
            # Log-likelihood under each coin hypothesis
            ll_A = get_log_likelihood(exp, (current_theta_A, 1 - current_theta_A))
            ll_B = get_log_likelihood(exp, (current_theta_B, 1 - current_theta_B))
            
            # Convert to probabilities (soft assignment weights)
            # P(coin A | data) proportional to P(data | coin A)
            weight_A = exp(ll_A) / (exp(ll_A) + exp(ll_B))
            weight_B = exp(ll_B) / (exp(ll_A) + exp(ll_B))
            
            # Expected counts for each coin
            expected_A[i] = weight_A * np.array(exp)
            expected_B[i] = weight_B * np.array(exp)
        
        # M-STEP: Update parameters using weighted maximum likelihood
        # New theta = (weighted sum of heads) / (weighted sum of all tosses)
        new_theta_A = expected_A[:, 0].sum() / expected_A.sum()
        new_theta_B = expected_B[:, 0].sum() / expected_B.sum()
        
        theta_A.append(new_theta_A)
        theta_B.append(new_theta_B)
        
        # Check convergence
        improvement = max(abs(new_theta_A - current_theta_A), 
                          abs(new_theta_B - current_theta_B))
        iteration += 1
    
    return np.array(theta_A), np.array(theta_B)


# The observed data: head counts from 5 experiments
head_counts = np.array([5, 9, 8, 4, 7])

# Run EM
theta_A, theta_B = em_coin_toss(head_counts)

print(f"Final estimates after {len(theta_A)-1} iterations:")
print(f"  Coin A bias (P(heads)): {theta_A[-1]:.4f}")
print(f"  Coin B bias (P(heads)): {theta_B[-1]:.4f}")

### Visualise the Convergence

Watch how EM iteratively refines its estimates:

In [ ]:
plt.figure(figsize=(10, 5))

# Plot convergence
plt.subplot(1, 2, 1)
iterations = range(len(theta_A))
plt.plot(iterations, theta_A, 'b-o', label='Coin A', markersize=8)
plt.plot(iterations, theta_B, 'r--s', label='Coin B', markersize=8)
plt.axhline(y=0.80, color='b', linestyle=':', alpha=0.5, label='True A (0.80)')
plt.axhline(y=0.45, color='r', linestyle=':', alpha=0.5, label='True B (0.45)')
plt.xlabel('Iteration')
plt.ylabel('Estimated P(heads)')
plt.title('EM Convergence')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot the data
plt.subplot(1, 2, 2)
experiments = ['Exp 1', 'Exp 2', 'Exp 3', 'Exp 4', 'Exp 5']
heads = [5, 9, 8, 4, 7]
tails = [5, 1, 2, 6, 3]

x = np.arange(len(experiments))
width = 0.35

plt.bar(x - width/2, heads, width, label='Heads', color='gold')
plt.bar(x + width/2, tails, width, label='Tails', color='silver')
plt.xlabel('Experiment')
plt.ylabel('Count')
plt.title('Observed Data')
plt.xticks(x, experiments)
plt.legend()

plt.tight_layout()
plt.show()

print("\nGround truth (if we knew which coin was used):")
print("  Experiments 2, 3, 5 used Coin A: (9+8+7)/(30) = 0.80")
print("  Experiments 1, 4 used Coin B: (5+4)/(20) = 0.45")

**You just ran EM!** Notice how the algorithm converged to estimates very close to the true values (0.80 and 0.45), even though it had no idea which coin was used for each experiment.

Now let's understand how it works.

## 3. What Just Happened?

The EM algorithm alternates between two steps:

### E-Step (Expectation): "Which coin was it, probably?"

Given our current guesses for the coin biases, we ask: "How likely is each experiment to have come from Coin A vs Coin B?"

For example, Experiment 2 (9 heads, 1 tail) is much more likely to have come from a coin biased toward heads. These "soft assignments" are weights between 0 and 1.

### M-Step (Maximisation): "Update our estimates"

Using these soft assignments as weights, we compute new parameter estimates. If Experiment 2 is 90% likely to be from Coin A, then 90% of its heads count goes toward our estimate of Coin A's bias.

Let's trace through one iteration manually:

In [ ]:
# Manual walkthrough of one iteration
theta_A_current = 0.6  # Our initial guess
theta_B_current = 0.5

experiments = [(5, 5), (9, 1), (8, 2), (4, 6), (7, 3)]

print("E-STEP: Computing soft assignments")
print("="*60)

weights_A = []
weights_B = []

for i, (heads, tails) in enumerate(experiments):
    # P(data | coin A) - assuming binomial distribution
    ll_A = get_log_likelihood((heads, tails), (theta_A_current, 1 - theta_A_current))
    ll_B = get_log_likelihood((heads, tails), (theta_B_current, 1 - theta_B_current))
    
    # Normalise to get probabilities
    prob_A = exp(ll_A)
    prob_B = exp(ll_B)
    weight_A = prob_A / (prob_A + prob_B)
    weight_B = prob_B / (prob_A + prob_B)
    
    weights_A.append(weight_A)
    weights_B.append(weight_B)
    
    print(f"Experiment {i+1} ({heads}H, {tails}T):")
    print(f"  P(data|A) ~ {prob_A:.6f}, P(data|B) ~ {prob_B:.6f}")
    print(f"  Weight A: {weight_A:.3f}, Weight B: {weight_B:.3f}")
    print()

In [ ]:
print("M-STEP: Updating parameters with weighted counts")
print("="*60)

# Weighted counts for Coin A
weighted_heads_A = sum(w * h for w, (h, t) in zip(weights_A, experiments))
weighted_total_A = sum(w * (h + t) for w, (h, t) in zip(weights_A, experiments))

# Weighted counts for Coin B  
weighted_heads_B = sum(w * h for w, (h, t) in zip(weights_B, experiments))
weighted_total_B = sum(w * (h + t) for w, (h, t) in zip(weights_B, experiments))

new_theta_A = weighted_heads_A / weighted_total_A
new_theta_B = weighted_heads_B / weighted_total_B

print(f"Coin A: weighted heads = {weighted_heads_A:.2f}, weighted total = {weighted_total_A:.2f}")
print(f"  New theta_A = {weighted_heads_A:.2f} / {weighted_total_A:.2f} = {new_theta_A:.4f}")
print()
print(f"Coin B: weighted heads = {weighted_heads_B:.2f}, weighted total = {weighted_total_B:.2f}")
print(f"  New theta_B = {weighted_heads_B:.2f} / {weighted_total_B:.2f} = {new_theta_B:.4f}")
print()
print(f"Change from initial: A: {theta_A_current:.2f} -> {new_theta_A:.4f}, B: {theta_B_current:.2f} -> {new_theta_B:.4f}")

### The Key Insight

Notice how experiments with lots of heads (2, 3, 5) get higher weights for Coin A, while experiments with fewer heads (1, 4) get higher weights for Coin B.

This creates a **chicken-and-egg** situation that EM resolves iteratively:
- To assign experiments to coins, we need to know the coin biases
- To estimate coin biases, we need to know which coin was used

EM breaks this cycle by using **soft assignments** (probabilities) instead of hard assignments, and iterating until convergence.

## 4. Going Deeper: The Mathematics

### The Likelihood Function

For each experiment with $h$ heads and $t$ tails, the probability under a coin with bias $\theta$ is:

$$P(h, t | \theta) = \binom{h+t}{h} \theta^h (1-\theta)^t$$

In log form (which is more numerically stable):

$$\log P(h, t | \theta) = \log\binom{h+t}{h} + h \log\theta + t \log(1-\theta)$$

### The E-Step

For each experiment $i$, we compute the posterior probability that it came from Coin A:

$$w_i^{(A)} = \frac{P(data_i | \theta_A)}{P(data_i | \theta_A) + P(data_i | \theta_B)}$$

This is just Bayes' rule with equal priors (we assume 50-50 chance of picking either coin).

### The M-Step

The new parameter estimates are weighted MLEs:

$$\theta_A^{new} = \frac{\sum_i w_i^{(A)} \cdot h_i}{\sum_i w_i^{(A)} \cdot (h_i + t_i)}$$

Think of it as: "the fraction of heads, but each experiment's contribution is weighted by how likely it was to come from this coin."

### Why Does EM Converge?

EM has a beautiful guarantee: **each iteration increases (or maintains) the log-likelihood**.

Let's verify this:

In [ ]:
def compute_total_log_likelihood(head_counts, theta_A, theta_B, n_tosses=10):
    """Compute total log-likelihood of data under current parameters."""
    experiments = list(zip(head_counts, n_tosses - head_counts))
    total_ll = 0
    
    for exp in experiments:
        # P(data) = P(data|A) * P(A) + P(data|B) * P(B)
        # With equal priors: P(data) = 0.5 * (P(data|A) + P(data|B))
        ll_A = get_log_likelihood(exp, (theta_A, 1 - theta_A))
        ll_B = get_log_likelihood(exp, (theta_B, 1 - theta_B))
        
        # Log of sum requires log-sum-exp trick for numerical stability
        max_ll = max(ll_A, ll_B)
        total_ll += max_ll + log(exp(ll_A - max_ll) + exp(ll_B - max_ll)) - log(2)
    
    return total_ll

# Compute log-likelihood at each iteration
log_likelihoods = []
for i in range(len(theta_A)):
    ll = compute_total_log_likelihood(head_counts, theta_A[i], theta_B[i])
    log_likelihoods.append(ll)

plt.figure(figsize=(8, 4))
plt.plot(range(len(log_likelihoods)), log_likelihoods, 'g-o', markersize=8)
plt.xlabel('Iteration')
plt.ylabel('Log-Likelihood')
plt.title('EM Increases Log-Likelihood at Each Step')
plt.grid(True, alpha=0.3)
plt.show()

print("Log-likelihood progression:")
for i, ll in enumerate(log_likelihoods):
    change = f"(+{ll - log_likelihoods[i-1]:.4f})" if i > 0 else ""
    print(f"  Iteration {i}: {ll:.4f} {change}")

### Local vs Global Optima

**Important caveat:** EM is guaranteed to find a local optimum, but not necessarily the global optimum. The final result depends on the initial values.

Let's see what happens with different initialisations:

In [ ]:
# Try different initialisations
init_pairs = [
    (0.6, 0.5),   # Original
    (0.5, 0.5),   # Symmetric start
    (0.9, 0.1),   # Extreme start
    (0.4, 0.7),   # Swapped bias direction
]

plt.figure(figsize=(12, 4))

for idx, (init_A, init_B) in enumerate(init_pairs):
    theta_A, theta_B = em_coin_toss(head_counts, theta_A_init=init_A, theta_B_init=init_B)
    
    plt.subplot(1, 4, idx + 1)
    plt.plot(theta_A, 'b-o', label='A', markersize=4)
    plt.plot(theta_B, 'r-s', label='B', markersize=4)
    plt.axhline(y=0.80, color='b', linestyle=':', alpha=0.5)
    plt.axhline(y=0.45, color='r', linestyle=':', alpha=0.5)
    plt.xlabel('Iteration')
    plt.ylabel('Theta')
    plt.title(f'Init: A={init_A}, B={init_B}')
    plt.legend(fontsize=8)
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: With symmetric initialisation (0.5, 0.5), the coins remain indistinguishable!")
print("This is a saddle point - any asymmetric perturbation would break the symmetry.")

### When NOT to Use EM

EM is powerful but has limitations:

1. **Symmetry issues**: If parameters are interchangeable (like our two coins), symmetric initialisation can cause problems
2. **Local optima**: Multiple random initialisations are often needed
3. **Slow convergence**: Can be slow near the optimum (linear convergence rate)
4. **Model misspecification**: If the data doesn't match your model assumptions, EM will still converge - to the wrong answer

**Alternatives:**
- Variational Inference (for Bayesian posteriors)
- MCMC (for full posterior distributions, not just point estimates)
- Gradient-based optimisation (if the likelihood is differentiable)

## 5. Exercises

Try these modifications to deepen your understanding:

In [ ]:
# Exercise 1: What happens with more data?
# Generate more experiments and see if EM converges faster/more accurately

# True parameters
true_theta_A = 0.8
true_theta_B = 0.45

np.random.seed(42)
n_experiments = 20  # Try changing this!
n_tosses = 10

# Generate data
coins_used = np.random.choice([0, 1], size=n_experiments)  # 0 = A, 1 = B
generated_heads = np.array([
    np.random.binomial(n_tosses, true_theta_A if c == 0 else true_theta_B)
    for c in coins_used
])

theta_A, theta_B = em_coin_toss(generated_heads)
print(f"Generated {n_experiments} experiments")
print(f"True values: A={true_theta_A}, B={true_theta_B}")
print(f"EM estimates: A={theta_A[-1]:.4f}, B={theta_B[-1]:.4f}")

In [ ]:
# Exercise 2: Extend to 3 coins
# Can you modify the algorithm to handle 3 coins instead of 2?
# Hint: The E-step needs to compute 3 weights that sum to 1

# Your code here:
# def em_three_coins(head_counts, ...):
#     ...

In [ ]:
# Exercise 3: Add a prior on the coin selection
# What if we know Coin A is used more often (e.g., 70% of the time)?
# Modify the E-step to incorporate P(coin A) != P(coin B)

# Your code here:
# def em_coin_toss_with_prior(head_counts, prior_A=0.7, ...):
#     ...

## 6. The Foundations

### Historical Context

The EM algorithm was formalised by Arthur Dempster, Nan Laird, and Donald Rubin in their seminal 1977 paper:

> Dempster, A.P., Laird, N.M. and Rubin, D.B. (1977). "Maximum Likelihood from Incomplete Data via the EM Algorithm". *Journal of the Royal Statistical Society, Series B*. 39 (1): 1–38.

The algorithm itself had been used in various forms before, but this paper provided the general framework and proved its convergence properties.

### Why It Matters

EM is foundational to many modern ML algorithms:
- **Gaussian Mixture Models (GMMs)** - clustering with soft assignments
- **Hidden Markov Models (HMMs)** - the Baum-Welch algorithm is EM
- **Topic Models (LDA)** - variational EM is used
- **Missing data imputation** - EM handles missing values naturally

### Further Reading

- **The original paper** (1977) - Dense but rewarding; focus on Sections 1-3
- **Stanford EM tutorial** - The example in this notebook comes from [this accessible tutorial](http://algorithmicalley.com/archive/2013/03/29/the-expectation-maximization-algorithm.aspx)
- **Bishop's PRML Chapter 9** - Excellent coverage of EM and its applications
- **Next: Gaussian Mixture Models** - Apply EM to continuous data clustering

---

**Author:** Dr. Berkan Sesen | [sesen.ai](https://sesen.ai)

**Companion blog post:** [The EM Algorithm: An Intuitive Guide](https://sesen.ai/blog/em-algorithm-coin-toss-intuitive-guide)